# Corpus Re-embedding + Retrieval Experiments

Encodes the 374k-document corpus with a biomedical embedding model and evaluates
against MiniLM-L6-v2 (baseline). Run on Colab A100 or L4.

**Steps (run top to bottom):**
1. GPU check → Install → HF token → Mount Drive → Config
2. Clone repo
3. Optionally encode corpus (`RUN_ENCODE = True`)
4. Load data (embeddings, index, texts, topics, qrels)
5. Define encode functions
6. Dense eval (cosine retrieval: MiniLM vs new model)
7. Hybrid eval (dense + BM25 via RRF: MiniLM vs new model)
8. Pipeline eval (sim→SVM→SciBERT with new embeddings)
9. Summary table (persisted across sessions on Drive)

**Recommended models:**
- `--medcpt` — `ncbi/MedCPT-Article-Encoder` (768-dim, biomedical specialist, asymmetric)
- `NeuML/pubmedbert-base-embeddings` — 768-dim, sentence-transformers compatible
- `BAAI/bge-base-en-v1.5` — 768-dim, top MTEB scores


In [ ]:
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — encoding 374k docs will be very slow')


In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q sentence-transformers datasets scikit-learn transformers tqdm lxml rank-bm25 huggingface_hub
!pip install -q sympy==1.13.1


In [ ]:
import os
os.environ['HF_TOKEN'] = ''  # paste your WRITE token (huggingface.co/settings/tokens)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# ===== PATHS =====================================================================
DATA_ROOT = '/content/drive/MyDrive/ct_data23'   # adjust if your Drive path differs
REPO      = '/content/ctmatch'

# ===== EMBEDDING MODEL ===========================================================
# Option A: MedCPT  (recommended — biomedical specialist, asymmetric encoders)
EMBED_CMD   = '--medcpt'
EMB_FILE    = 'doc_embeddings_medcpt.npy'
QUERY_MODEL = 'ncbi/MedCPT-Query-Encoder'

# Option B: PubMedBERT
# EMBED_CMD   = '--model NeuML/pubmedbert-base-embeddings'
# EMB_FILE    = 'doc_embeddings_pubmedbert-base-embeddings.npy'
# QUERY_MODEL = 'NeuML/pubmedbert-base-embeddings'

# Option C: BGE base
# EMBED_CMD   = '--model BAAI/bge-base-en-v1.5'
# EMB_FILE    = 'doc_embeddings_bge-base-en-v1.5.npy'
# QUERY_MODEL = 'BAAI/bge-base-en-v1.5'

BATCH_SIZE = 256   # reduce to 128 if OOM

# ===== DATASETS ==================================================================
USE_TREC_2021 = True
USE_TREC_2022 = True
USE_KZ        = True
MAX_TOPICS    = None    # set e.g. 75 for a quick test run

# ===== RUN FLAGS =================================================================
RUN_ENCODE        = False   # True → re-encode corpus (~1hr on A100)
FORCE_ENCODE      = False   # True → overwrite existing embeddings without asking
PUSH_TO_HF        = True    # push new embeddings to HF hub after encoding
RUN_DENSE_EVAL    = True
RUN_HYBRID_EVAL   = True
RUN_PIPELINE_EVAL = True

# ===== DERIVED PATHS (auto-set from DATA_ROOT above) =============================
_TREC = os.path.join(DATA_ROOT, 'evaluation', 'trec_data')
_KZ   = os.path.join(DATA_ROOT, 'evaluation', 'kz_data')

TREC21_TOPIC_PATH = os.path.join(_TREC, 'trec_21_topics.xml')
TREC21_REL_PATH   = os.path.join(_TREC, 'trec_21_judgments.txt')
TREC22_TOPIC_PATH = os.path.join(_TREC, 'topics2022.xml')
TREC22_REL_PATH   = os.path.join(_TREC, 'qrels2022.txt')
KZ_TOPIC_PATH     = os.path.join(_KZ,   'topics-2014_2015-description.topics')
KZ_REL_PATH       = os.path.join(_KZ,   'qrels-clinical_trials.txt')
RESULTS_FILE      = os.path.join(DATA_ROOT, 'eval_results.json')

print(f'Data root : {DATA_ROOT}')
print(f'Embedding : {EMBED_CMD}  →  {EMB_FILE}')
print(f'Query enc : {QUERY_MODEL}')
print(f'Datasets  : TREC2021={USE_TREC_2021}  TREC2022={USE_TREC_2022}  KZ={USE_KZ}  MAX={MAX_TOPICS}')
print(f'Flags     : encode={RUN_ENCODE}  force={FORCE_ENCODE}  push={PUSH_TO_HF}')
print(f'Eval      : dense={RUN_DENSE_EVAL}  hybrid={RUN_HYBRID_EVAL}  pipeline={RUN_PIPELINE_EVAL}')


In [ ]:
import subprocess
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/semajyllek/ctmatch.git', REPO], check=True)
    print(f'Cloned to {REPO}')
else:
    print(f'Repo already at {REPO}')


In [ ]:
import json

def load_results():
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            return json.load(f)
    return {}

def save_result(metrics: dict):
    all_r = load_results()
    all_r[metrics['Model']] = metrics
    with open(RESULTS_FILE, 'w') as f:
        json.dump(all_r, f, indent=2)
    print(f"  saved \u2192 {RESULTS_FILE}")

all_results = load_results()
print(f'Loaded {len(all_results)} existing result(s): {list(all_results.keys())}')


In [ ]:
if RUN_ENCODE:
    emb_candidates = [os.path.join(REPO, EMB_FILE), os.path.join(DATA_ROOT, EMB_FILE)]
    existing = [p for p in emb_candidates if os.path.exists(p)]
    if existing and not FORCE_ENCODE:
        raise RuntimeError(
            f'Embeddings already exist at: {existing[0]}\n'
            f'Set FORCE_ENCODE=True in the Config cell to overwrite.'
        )
    !cd {REPO} && python scripts/build_embeddings.py {EMBED_CMD} --output {EMB_FILE} --batch-size {BATCH_SIZE}
    if PUSH_TO_HF:
        from huggingface_hub import HfApi
        HfApi().upload_file(
            path_or_fileobj=os.path.join(REPO, EMB_FILE),
            path_in_repo=EMB_FILE,
            repo_id='semaj83/ctmatch_ir',
            repo_type='dataset',
        )
        print(f'Pushed {EMB_FILE} → semaj83/ctmatch_ir')
else:
    print('RUN_ENCODE=False — skipping encoding step')


In [ ]:
import sys
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

sys.path.insert(0, f'{REPO}/src')

from ctmatch.evaluation.eval_utils import (
    get_trec_topic2text, get_kz_topic2text,
    calc_first_positive_rank, calc_f1, calc_ndcg,
)

HF_ROOT = 'semaj83/ctmatch_ir'

# --- corpus index ---
print('Loading index2docid...')
idx_ds = load_dataset(HF_ROOT, data_files='index2docid.txt', split='train')
index2docid = pd.DataFrame(idx_ds)
print(f'  {len(index2docid)} docs')

# --- MiniLM baseline embeddings (384-dim, stored as CSV on HF) ---
print('Loading MiniLM baseline embeddings from HF...')
old_ds = load_dataset(HF_ROOT, data_files='doc_embeddings.txt', split='train')
old_emb = np.array([np.fromstring(r['text'], sep=',', dtype=np.float32) for r in old_ds])
print(f'  MiniLM: {old_emb.shape}')

# --- new embeddings (look in DATA_ROOT first, then REPO if just encoded) ---
new_emb_path = os.path.join(DATA_ROOT, EMB_FILE)
if not os.path.exists(new_emb_path):
    new_emb_path = os.path.join(REPO, EMB_FILE)
if not os.path.exists(new_emb_path):
    raise FileNotFoundError(
        f'{EMB_FILE} not found in {DATA_ROOT} or {REPO}. '
        f'Set RUN_ENCODE=True to build it.'
    )
new_emb = np.load(new_emb_path)
print(f'  {EMB_FILE}: {new_emb.shape}')

# --- doc texts for BM25 hybrid eval ---
print('Loading doc texts via DataPrep (needed for BM25)...')
from ctmatch.data.dataprep import DataPrep
from ctmatch.config import PipeConfig
_dp = DataPrep(PipeConfig(ir_setup=True))
doc_texts = [v[0] for v in _dp.doc_texts_df.values]
del _dp
print(f'  {len(doc_texts)} doc texts')

# --- topics + qrels ---
topic2text, rel_dict = {}, {}
for kind, tpath, rpath, flag in [
    ('trec', TREC21_TOPIC_PATH, TREC21_REL_PATH, USE_TREC_2021),
    ('trec', TREC22_TOPIC_PATH, TREC22_REL_PATH, USE_TREC_2022),
    ('kz',   KZ_TOPIC_PATH,     KZ_REL_PATH,     USE_KZ),
]:
    if not flag:
        continue
    if os.path.exists(tpath):
        fn = get_trec_topic2text if kind == 'trec' else get_kz_topic2text
        topic2text.update(fn(tpath))
    else:
        print(f'WARNING: missing {tpath}')
    if os.path.exists(rpath):
        with open(rpath) as _f:
            for line in _f:
                tid, _, did, rel = line.split()
                rel_dict.setdefault(tid, {})[did] = int(rel)
    else:
        print(f'WARNING: missing {rpath}')

print(f'\n{len(topic2text)} topics loaded, {len(rel_dict)} with judgments')

def get_doc_indices(doc_ids):
    out = []
    for d in doc_ids:
        rows = np.where(index2docid['text'] == d)[0]
        if len(rows):
            out.append(int(rows[0]))
    return out


In [ ]:
import torch
from sentence_transformers import SentenceTransformer

# MiniLM baseline encoder (384-dim, symmetric)
_old_st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def old_encode(text):
    return _old_st.encode([text])[0]

# New model encoder
if 'medcpt' in EMBED_CMD.lower():
    # MedCPT is asymmetric: Article Encoder for corpus, Query Encoder at query time
    from transformers import AutoTokenizer, AutoModel
    _device = 'cuda' if torch.cuda.is_available() else 'cpu'
    _qt = AutoTokenizer.from_pretrained('ncbi/MedCPT-Query-Encoder')
    _qm = AutoModel.from_pretrained('ncbi/MedCPT-Query-Encoder').eval().to(_device)
    def new_encode(text):
        enc = _qt([text], return_tensors='pt', truncation=True, max_length=512, padding=True)
        enc = {k: v.to(_device) for k, v in enc.items()}
        with torch.no_grad():
            return _qm(**enc).last_hidden_state[:, 0, :].cpu().numpy()[0]
else:
    _new_st = SentenceTransformer(QUERY_MODEL)
    def new_encode(text):
        return _new_st.encode([text])[0]

print('Encode functions ready')


## Dense Eval — cosine retrieval on judged pool

In [ ]:
def cosine_rank(topic_emb, emb_matrix, doc_set):
    vecs = emb_matrix[doc_set]
    sims = vecs @ topic_emb / (np.linalg.norm(vecs, axis=1) * np.linalg.norm(topic_emb) + 1e-9)
    return [doc_set[i] for i in np.argsort(-sims)]

def eval_dense(encode_fn, emb_matrix, name):
    fprs, frrs, f1s, ndcgs = [], [], [], []
    topics = list(topic2text.items())
    if MAX_TOPICS:
        topics = topics[:MAX_TOPICS]
    for topic_id, topic_text in tqdm(topics, desc=name):
        if topic_id not in rel_dict:
            continue
        doc_set = get_doc_indices(list(rel_dict[topic_id].keys()))
        if not doc_set:
            continue
        ranked = cosine_rank(encode_fn(topic_text), emb_matrix, doc_set)
        ranked_ids = [index2docid.iloc[i]['text'] for i in ranked]
        fpr, frr = calc_first_positive_rank(ranked_ids, rel_dict[topic_id])
        fprs.append(fpr); frrs.append(frr)
        f1s.append(calc_f1(ranked_ids, rel_dict[topic_id]))
        ndcgs.append(calc_ndcg(ranked_ids, rel_dict[topic_id], k=10))
    return {
        'Model':   name,
        'NDCG@10': round(float(np.mean(ndcgs)), 4),
        'MRR':     round(float(np.mean(frrs)), 4),
        'F1':      round(float(np.mean(f1s)), 4),
        'FPR':     round(float(np.mean(fprs)), 4),
    }

if RUN_DENSE_EVAL:
    old_res = eval_dense(old_encode, old_emb, 'MiniLM-L6-v2')
    save_result(old_res)
    new_res = eval_dense(new_encode, new_emb, EMB_FILE)
    save_result(new_res)
    all_results.update({old_res['Model']: old_res, new_res['Model']: new_res})
    pd.DataFrame([old_res, new_res]).set_index('Model').round(4)


## Hybrid Eval — dense + BM25 via Reciprocal Rank Fusion

In [ ]:
from rank_bm25 import BM25Okapi

def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def eval_hybrid(encode_fn, emb_matrix, name, rrf_k=60):
    fprs, frrs, f1s, ndcgs = [], [], [], []
    topics = list(topic2text.items())
    if MAX_TOPICS:
        topics = topics[:MAX_TOPICS]
    for topic_id, topic_text in tqdm(topics, desc=name):
        if topic_id not in rel_dict:
            continue
        doc_set = get_doc_indices(list(rel_dict[topic_id].keys()))
        if not doc_set:
            continue
        dense_ranked = cosine_rank(encode_fn(topic_text), emb_matrix, doc_set)
        subset_texts = [doc_texts[i] for i in doc_set]
        bm25 = BM25Okapi([t.split() for t in subset_texts])
        bm25_ranked = [doc_set[i] for i in np.argsort(-bm25.get_scores(topic_text.split()))]
        fused = rrf([dense_ranked, bm25_ranked], k=rrf_k)
        ranked_ids = [index2docid.iloc[i]['text'] for i in fused]
        fpr, frr = calc_first_positive_rank(ranked_ids, rel_dict[topic_id])
        fprs.append(fpr); frrs.append(frr)
        f1s.append(calc_f1(ranked_ids, rel_dict[topic_id]))
        ndcgs.append(calc_ndcg(ranked_ids, rel_dict[topic_id], k=10))
    return {
        'Model':   name,
        'NDCG@10': round(float(np.mean(ndcgs)), 4),
        'MRR':     round(float(np.mean(frrs)), 4),
        'F1':      round(float(np.mean(f1s)), 4),
        'FPR':     round(float(np.mean(fprs)), 4),
    }

if RUN_HYBRID_EVAL:
    hybrid_old = eval_hybrid(old_encode, old_emb, 'MiniLM-L6-v2 + BM25')
    save_result(hybrid_old)
    hybrid_new = eval_hybrid(new_encode, new_emb, f'{EMB_FILE} + BM25')
    save_result(hybrid_new)
    all_results.update({hybrid_old['Model']: hybrid_old, hybrid_new['Model']: hybrid_new})
    pd.DataFrame(list(all_results.values())).set_index('Model').round(4)


## Pipeline Eval — full sim\u2192SVM\u2192SciBERT with new embeddings

In [ ]:
if RUN_PIPELINE_EVAL:
    from ctmatch.evaluation.evaluator import EvaluatorConfig, Evaluator
    from ctmatch.config import PipeConfig
    from ctmatch.matching.pipeline import CTMatch

    pipe_config = PipeConfig(
        ir_setup=True,
        embedding_file=EMB_FILE,
        embedding_model_checkpoint=QUERY_MODEL,
        filters=['sim', 'svm', 'classifier'],
    )
    ctm = CTMatch(pipe_config)

    trec_topic_paths = [p for p, flag in [(TREC21_TOPIC_PATH, USE_TREC_2021),
                                           (TREC22_TOPIC_PATH, USE_TREC_2022)]
                        if flag and os.path.exists(p)]
    rel_paths = [p for p, flag in [(TREC21_REL_PATH, USE_TREC_2021),
                                    (TREC22_REL_PATH, USE_TREC_2022),
                                    (KZ_REL_PATH,     USE_KZ)] if flag]

    eval_config = EvaluatorConfig(
        rel_paths=rel_paths,
        trec_topic_path=trec_topic_paths or None,
        kz_topic_path=KZ_TOPIC_PATH if USE_KZ else None,
        max_topics=MAX_TOPICS,
        filters=['sim', 'svm', 'classifier'],
        ctm=ctm,
    )
    evaluator = Evaluator(eval_config)
    output_file = f'eval_predictions_{EMB_FILE.replace(".npy", "")}.jsonl'
    res = evaluator.evaluate_detailed(output_file)

    pipeline_res = {
        'Model':   f'{EMB_FILE} pipeline',
        'NDCG@10': round(res.get('ndcg@10', 0.0), 4),
        'MRR':     round(res.get('mean_mrr', 0.0), 4),
        'F1':      round(res.get('mean_f1', 0.0), 4),
        'FPR':     round(res.get('mean_fpr', 0.0), 4),
    }
    save_result(pipeline_res)
    all_results[pipeline_res['Model']] = pipeline_res
    print(pipeline_res)


## Summary — all results (loaded from Drive, persists across sessions)

In [ ]:
all_results = load_results()
if all_results:
    pd.DataFrame(list(all_results.values())).set_index('Model').round(4)
else:
    print('No results yet \u2014 run eval cells above')
